# Preprocessing

This notebook uses the original CSV as the main dataset because it contains the raw fields needed for cleaning, feature selection, and feature-group experiments. The `_ML.csv` file is useful as a reference, but it is already transformed and has a different row count and target distribution.

## Load Data

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/youtube_shorts_tiktok_trends_2025.csv")
df_ml = pd.read_csv("../data/youtube_shorts_tiktok_trends_2025.csv_ML.csv")

print("Original shape:", df.shape)
print("ML shape:", df_ml.shape)

df.head()

Original shape: (48079, 58)
ML shape: (50000, 32)


,platform,country,region,language,category,hashtag,title_keywords,author_handle,sound_type,music_track,...,traffic_source,is_weekend,row_id,engagement_total,like_rate,dislike_rate,engagement_per_1k,engagement_like_rate,engagement_comment_rate,engagement_share_rate
0,TikTok,Jp,Asia,ja,Gaming,#Lifestyle,Night Routine — College,NextVision,trending,8bit loop,...,External,1,2e681528d17a1fe1986857942536ec27,30317,0.086159,0.004004,120.069,0.086159,0.012555,0.007830
1,TikTok,Se,Europe,sv,Food,#Sports,Morning Routine — College,DailyVlogsDiego,trending,Street vibe,...,Search,0,2e35fa0b2978b9cae635839c1d4e9e74,30577,0.085298,0.002421,113.005,0.085298,0.007850,0.007791
2,TikTok,Za,Africa,en,Art,#Workout,Night Routine — College,BeyondHub,licensed,Gallery pad,...,External,1,0d88a011235a82244995ef52961f9502,503,0.049154,0.001625,68.111,0.049154,0.004469,0.005146
3,TikTok,Kr,Asia,ko,News,#Esports,Best Settings for Fortnite,NextHub,original,Neutral piano,...,Search,1,e15cff7621ed3f9eb9d2c97c841be0f3,7828,0.086257,0.003164,108.156,0.086257,0.011205,0.005292
4,TikTok,Au,Oceania,en,Beauty,#Comedy,When your friend is Beginners,LucasOfficial,licensed,Soft glam loop,...,ForYou,1,d696b4f0a50ea70e7cb5021be7e198ec,1171,0.051441,0.001175,72.400,0.051441,0.004204,0.004142


## Compare Original vs ML Version

The row count and label distribution are different, so we should not treat these as interchangeable versions of the same table.

In [4]:
print("Original target distribution")
display(df["trend_label"].value_counts(normalize=True).sort_index())

print("ML target distribution")
display(df_ml["trend_label"].value_counts(normalize=True).sort_index())

ml_only_columns = sorted(set(df_ml.columns) - set(df.columns))
original_only_columns = sorted(set(df.columns) - set(df_ml.columns))

print("Columns only in ML version:")
display(ml_only_columns)

print("Columns only in original:")
display(original_only_columns)

Original target distribution


trend_label
declining    0.249652
rising       0.251711
seasonal     0.252626
stable       0.246012
Name: proportion, dtype: float64

ML target distribution


trend_label
declining    0.10196
rising       0.25000
seasonal     0.09412
stable       0.55392
Name: proportion, dtype: float64

Columns only in ML version:


['category_cat',
 'comment_rate',
 'comment_rate_log',
 'creator_tier_cat',
 'device_brand_cat',
 'language_cat',
 'like_hashtag_interaction',
 'like_rate_log',
 'likes_per_day',
 'platform_cat',
 'region_cat',
 'rel_combo',
 'rel_like',
 'rel_share',
 'richness_traffic_interaction',
 'share_hashtag_interaction',
 'share_rate_log',
 'text_richness',
 'title_len',
 'traffic_source_cat',
 'views_per_day',
 'weekend_hashtag_boost']

Columns only in original:


['author_handle',
 'avg_watch_time_sec',
 'comment_ratio',
 'comments',
 'completion_rate',
 'country',
 'creator_avg_views',
 'device_type',
 'dislike_rate',
 'dislikes',
 'duration_sec',
 'engagement_comment_rate',
 'engagement_like_rate',
 'engagement_per_1k',
 'engagement_rate',
 'engagement_share_rate',
 'engagement_total',
 'engagement_velocity',
 'event_season',
 'genre',
 'has_emoji',
 'hashtag',
 'is_weekend',
 'like_dislike_ratio',
 'likes',
 'music_track',
 'notes',
 'publish_date_approx',
 'publish_dayofweek',
 'publish_period',
 'row_id',
 'sample_comments',
 'save_rate',
 'saves',
 'season',
 'shares',
 'sound_type',
 'source_hint',
 'tags',
 'title',
 'title_keywords',
 'title_length',
 'trend_duration_days',
 'trend_type',
 'upload_hour',
 'views',
 'week_of_year',
 'year_month']

## Basic Data Checks

In [4]:
summary = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "n_unique": df.nunique(),
    }
)

print("Duplicate rows:", df.duplicated().sum())
if "row_id" in df.columns:
    print("Duplicate row_id:", df["row_id"].duplicated().sum())

display(summary)

Duplicate rows: 0
Duplicate row_id: 0


,dtype,missing,n_unique
platform,str,0,2
country,str,0,30
region,str,0,6
language,str,0,19
category,str,0,19
hashtag,str,0,41
title_keywords,str,0,137
author_handle,str,0,720
sound_type,str,0,3
music_track,str,0,61


## Clean Values

This keeps cleaning simple: trim whitespace, lowercase categorical text, parse the approximate publish date, and create simple date parts.

In [7]:
clean = df.copy()

text_columns = clean.select_dtypes(include=["object", "string"]).columns
for col in text_columns:
    clean[col] = clean[col].astype("string").str.strip().str.lower()

clean["publish_date_approx"] = pd.to_datetime(clean["publish_date_approx"], errors="coerce")
clean["publish_month"] = clean["publish_date_approx"].dt.month
clean["publish_day"] = clean["publish_date_approx"].dt.day

clean.head()

,platform,country,region,language,category,hashtag,title_keywords,author_handle,sound_type,music_track,...,row_id,engagement_total,like_rate,dislike_rate,engagement_per_1k,engagement_like_rate,engagement_comment_rate,engagement_share_rate,publish_month,publish_day
0,tiktok,jp,asia,ja,gaming,#lifestyle,night routine — college,nextvision,trending,8bit loop,...,2e681528d17a1fe1986857942536ec27,30317,0.086159,0.004004,120.069,0.086159,0.012555,0.007830,1,4
1,tiktok,se,europe,sv,food,#sports,morning routine — college,dailyvlogsdiego,trending,street vibe,...,2e35fa0b2978b9cae635839c1d4e9e74,30577,0.085298,0.002421,113.005,0.085298,0.007850,0.007791,1,1
2,tiktok,za,africa,en,art,#workout,night routine — college,beyondhub,licensed,gallery pad,...,0d88a011235a82244995ef52961f9502,503,0.049154,0.001625,68.111,0.049154,0.004469,0.005146,1,5
3,tiktok,kr,asia,ko,news,#esports,best settings for fortnite,nexthub,original,neutral piano,...,e15cff7621ed3f9eb9d2c97c841be0f3,7828,0.086257,0.003164,108.156,0.086257,0.011205,0.005292,1,3
4,tiktok,au,oceania,en,beauty,#comedy,when your friend is beginners,lucasofficial,licensed,soft glam loop,...,d696b4f0a50ea70e7cb5021be7e198ec,1171,0.051441,0.001175,72.400,0.051441,0.004204,0.004142,1,4


## Choose Columns

These drops are intentionally conservative. Text columns can be useful, but they need a separate NLP approach. Trend-duration fields are excluded from the first modelling frame because they describe the realised trend window.

In [10]:
TARGET = "trend_label"

text_id_or_notes_columns = [
    "row_id",
    "source_hint",
    "notes",
    "title",
    "title_keywords",
    "tags",
    "sample_comments",
    "author_handle",
    "music_track",
    "publish_date_approx",
    "year_month",
]

possible_leakage_columns = [
    "trend_duration_days",
    "trend_type",
    "engagement_velocity",
]

drop_columns = [TARGET] + text_id_or_notes_columns + possible_leakage_columns
drop_columns = [col for col in drop_columns if col in clean.columns]

X = clean.drop(columns=drop_columns)
y = clean[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (48079, 45)
y shape: (48079,)


## Feature Groups

Use these groups later for RQ experiments. For example, train one model with metadata only, then add temporal, creator, and engagement features.

In [11]:
feature_groups = {
    "metadata": [
        "platform",
        "country",
        "region",
        "language",
        "category",
        "hashtag",
        "sound_type",
        "device_type",
        "device_brand",
        "traffic_source",
    ],
    "temporal": [
        "week_of_year",
        "upload_hour",
        "publish_dayofweek",
        "publish_period",
        "event_season",
        "season",
        "is_weekend",
        "publish_month",
        "publish_day",
    ],
    "content_basic": [
        "genre",
        "duration_sec",
        "title_length",
        "has_emoji",
    ],
    "creator": [
        "creator_avg_views",
        "creator_tier",
    ],
    "engagement_observed": [
        "views",
        "likes",
        "comments",
        "shares",
        "saves",
        "dislikes",
        "engagement_total",
        "engagement_rate",
        "comment_ratio",
        "share_rate",
        "save_rate",
        "like_dislike_ratio",
        "like_rate",
        "dislike_rate",
        "engagement_per_1k",
        "engagement_like_rate",
        "engagement_comment_rate",
        "engagement_share_rate",
        "avg_watch_time_sec",
        "completion_rate",
    ],
}

feature_groups = {
    group: [col for col in cols if col in X.columns]
    for group, cols in feature_groups.items()
}

## Train / Validation / Test Split

The split is stratified so each set keeps roughly the same class balance.

In [13]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp,
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

display(y_train.value_counts(normalize=True).sort_index())
display(y_val.value_counts(normalize=True).sort_index())
display(y_test.value_counts(normalize=True).sort_index())

Train: (33655, 45) (33655,)
Validation: (7212, 45) (7212,)
Test: (7212, 45) (7212,)


trend_label
declining    0.249651
rising       0.251701
seasonal     0.252622
stable       0.246026
Name: proportion, dtype: Float64

trend_label
declining    0.249584
rising       0.251803
seasonal     0.252634
stable       0.245979
Name: proportion, dtype: Float64

trend_label
declining    0.249723
rising       0.251664
seasonal     0.252634
stable       0.245979
Name: proportion, dtype: Float64

## Save

In [14]:
train = X_train.copy()
train[TARGET] = y_train

validation = X_val.copy()
validation[TARGET] = y_val

test = X_test.copy()
test[TARGET] = y_test

train.to_csv("../data/processed/train.csv", index=False)
validation.to_csv("../data/processed/validation.csv", index=False)
test.to_csv("../data/processed/test.csv", index=False)

print("Saved files to", "../data/processed")

Saved files to ../data/processed


For **RQ1: influential factors**, the cleaned feature table lets us measure which variables are most useful for predicting `trend_label`. After training models, we can inspect coefficients, permutation importance, tree-based feature importance, or SHAP values. The feature groups above also make the interpretation easier because individual features can be discussed as metadata, temporal, creator, content, or engagement factors.

For **RQ2: feature group effectiveness**, the `feature_groups` dictionary gives a direct experiment plan. Train the same model repeatedly with different groups: metadata only, metadata plus temporal, metadata plus content, metadata plus creator, metadata plus engagement, and all selected features. Comparing validation accuracy, macro-F1, and class-level recall will show which groups improve prediction and whether engagement features dominate weaker metadata signals.

For **RQ3: model comparison**, the stratified train/validation/test split gives a fair basis for comparing traditional models and neural networks. Start with simple baselines such as majority class, logistic regression, decision tree, random forest, SVM, and gradient boosting. Then compare against an MLP using the same split and evaluation metrics. This keeps the model comparison focused on modelling ability rather than differences caused by inconsistent preprocessing.

A key caveat is that the original CSV and ML CSV are not interchangeable. The original has richer raw fields and balanced labels, but early checks suggest the provided `trend_label` may be difficult to predict from obvious raw features. The ML file has stronger engineered signal but a different row count and a different target distribution. In the report, we should explain why we use the original as the transparent preprocessing source, and optionally use the ML file as a comparison or sensitivity check.

Recommended next steps:

1. Build a baseline model using only the `metadata` group.
2. Add one feature group at a time and record validation macro-F1.
3. Check confusion matrices to see which trend labels are hardest to predict.
4. Train several model families on the same split for RQ3.
5. Use the held-out test set only once, after choosing the final feature set and model.